Run all other notebooks first

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

def clean_seq(seq):
    if pd.isna(seq): return ""
    return str(seq).replace("P", "").upper()

def analyze_fs_with_results_exclusion(root_dir="."):
    root = Path(root_dir)
    test_folder = root / "tests"
    results_folder = root / "results"
    
    # Create results folder if it doesn't exist
    results_folder.mkdir(exist_ok=True)
    
    if not test_folder.exists():
        print("Error: 'tests' folder not found.")
        return None

    # 1. Gather data files: 
    # Exclude files in 'tests' and files in any 'results' folder
    data_files = []
    for f in root.rglob("*.csv"):
        parts = f.parts
        if "tests" not in parts and "results" not in parts:
            data_files.append(f)

    # Group data files by name
    data_groups = {}
    for f in data_files:
        if f.name not in data_groups: 
            data_groups[f.name] = []
        data_groups[f.name].append(f)

    all_test_matrices = {}

    # 2. Iterate through each test file (the splits)
    for test_file_path in test_folder.glob("*.csv"):
        test_name = test_file_path.name
        print(f"Processing split: {test_name}...")
        
        try:
            test_df = pd.read_csv(test_file_path)
            if 'sequence' not in test_df.columns:
                print(f"  Skipping {test_name}: No 'sequence' column.")
                continue
            
            # Clean test sequences: Remove 'N', then strip 'P'
            test_sequences = [
                s.replace("P", "").upper() 
                for s in test_df['sequence'].dropna().astype(str).unique() 
                if "N" not in s.upper()
            ]
            
            if not test_sequences:
                print(f"  No valid sequences in {test_name} after cleaning.")
                continue

            split_results = []

            # 3. Process each dataset group against this test split
            for dataset_name, paths in data_groups.items():
                # TEMP: skip if isnt not SPROUT TODO: REMOE
                #if 'SPROUT' not in dataset_name:
                   #continue
                print(f"  Analyzing dataset: {dataset_name} in paths: {[str(p) for p in paths]}")
                try:
                    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
                    if 'sequence' not in df.columns: 
                        continue

                    # Pre-clean dataset sequences
                    df['clean_ds_seq'] = df['sequence'].apply(clean_seq)

                    # Logic: Sequence is identical OR test sequence is a suffix
                    def validate_and_match(ds_seq):
                        matches = [ts for ts in test_sequences if ds_seq.endswith(ts)]
                        if len(matches) > 1:
                            raise ValueError(f"Ambiguity: '{ds_seq}' matches multiple test suffixes: {matches}")
                        return matches[0] if len(matches) == 1 else None

                    df['matched_test_seq'] = df['clean_ds_seq'].apply(validate_and_match)
                    filtered_df = df[df['matched_test_seq'].notna()].copy()

                    if filtered_df.empty:
                        continue

                    # 4. Correlation Calculation
                    numeric_cols = [c for c in filtered_df.columns if c.lstrip('-').isdigit()]
                    if not numeric_cols: 
                        continue

                    # remove rows with all zeros in numeric columns
                    filtered_df = filtered_df[(filtered_df[numeric_cols] != 0).any(axis=1)]
                    if filtered_df.empty:
                        continue

                    # Normalize numeric rows to sum to 1
                    df_numeric = filtered_df[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
                    row_sums = df_numeric.sum(axis=1)
                    df_norm = df_numeric.div(row_sums.replace(0, 1), axis=0)

                    # Experimental FS: non-mod-3 sum (the real-world indel effect)
                    fs_target_cols = [c for c in numeric_cols if int(c) % 3 != 0]
                    experimental_fs = df_norm[fs_target_cols].sum(axis=1)

                    # Find all Predictors (columns with "FS" in name)
                    fs_predictors = [c for c in filtered_df.columns if "FS" in str(c)]
                    for predictor in fs_predictors:
                        pred_vals = pd.to_numeric(filtered_df[predictor], errors='coerce')
                        correlation = pred_vals.corr(experimental_fs)
                        
                        split_results.append({
                            'Dataset': dataset_name,
                            'Predictor': predictor,
                            'Correlation': correlation
                        })

                except ValueError as ve:
                    print(f"  [AMBIGUITY ERROR] in {dataset_name} for split {test_name}: {ve}")
                except Exception as e:
                    print(f"  Error in {dataset_name}: {e}")

            # 5. Pivot results and save to 'results' folder
            if split_results:
                matrix = pd.DataFrame(split_results).pivot(index='Dataset', columns='Predictor', values='Correlation')
                all_test_matrices[test_name] = matrix
                
                output_path = results_folder / f"matrix_{test_name}"
                matrix.to_csv(output_path)
                print(f"  Success: Saved to {output_path}")

        except Exception as e:
            print(f"Failed to process test split {test_name}: {e}")

    return all_test_matrices

# Run the analysis
analyze_fs_with_results_exclusion()

Processing split: test_CROTON_APINDEL.csv...
  Analyzing dataset: ALDIT_HAP1.csv in paths: ['FORECasT_preds\\ALDIT_HAP1.csv', 'inDelphi_preds\\ALDIT_HAP1.csv', 'Lindel_preds\\ALDIT_HAP1.csv']
  Analyzing dataset: ALDIT_Jurkat.csv in paths: ['FORECasT_preds\\ALDIT_Jurkat.csv', 'inDelphi_preds\\ALDIT_Jurkat.csv', 'Lindel_preds\\ALDIT_Jurkat.csv']
  Analyzing dataset: ALDIT_Jurkat_DNTTKO.csv in paths: ['FORECasT_preds\\ALDIT_Jurkat_DNTTKO.csv', 'inDelphi_preds\\ALDIT_Jurkat_DNTTKO.csv', 'Lindel_preds\\ALDIT_Jurkat_DNTTKO.csv']
  Analyzing dataset: ALDIT_K562.csv in paths: ['FORECasT_preds\\ALDIT_K562.csv', 'inDelphi_preds\\ALDIT_K562.csv', 'Lindel_preds\\ALDIT_K562.csv']
  Analyzing dataset: ALDIT_K562_DNTTOE.csv in paths: ['FORECasT_preds\\ALDIT_K562_DNTTOE.csv', 'inDelphi_preds\\ALDIT_K562_DNTTOE.csv', 'Lindel_preds\\ALDIT_K562_DNTTOE.csv']
  Analyzing dataset: FORECasT_BOB.csv in paths: ['FORECasT_preds\\FORECasT_BOB.csv', 'inDelphi_preds\\FORECasT_BOB.csv']
  Analyzing dataset: FORECa

{'test_CROTON_APINDEL.csv': Predictor                               FORECasT_FS_ratio  Lindel_FS_ratio  \
 Dataset                                                                      
 ALDIT_HAP1.csv                                   0.783772         0.753213   
 ALDIT_Jurkat.csv                                 0.549931         0.546051   
 ALDIT_Jurkat_DNTTKO.csv                          0.701661         0.716118   
 ALDIT_K562.csv                                   0.787591         0.784918   
 ALDIT_K562_DNTTOE.csv                            0.629731         0.631040   
 FORECasT_BOB.csv                                 0.690761              NaN   
 FORECasT_CHO.csv                                 0.689695              NaN   
 FORECasT_HAP1.csv                                0.778795              NaN   
 FORECasT_K562.csv                                0.798035              NaN   
 FORECasT_K562_2A_TREX2.csv                       0.632385              NaN   
 FORECasT_K562_TREX2.csv 

In [2]:

import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.metrics import roc_auc_score

def clean_seq(seq):
    """Standardizes sequences by removing 'P' and converting to uppercase."""
    if pd.isna(seq): return ""
    return str(seq).replace("P", "").upper()

def analyze_k562_auc_final(root_dir="."):
    root = Path(root_dir)
    test_folder = root / "tests"
    results_folder = root / "results"
    master_path = Path("../data/FORECasT_K562.csv")
    dataset_filename = master_path.name # "FORECasT_K562.csv"
    
    results_folder.mkdir(exist_ok=True)
    
    if not master_path.exists():
        print(f"CRITICAL ERROR: Master dataset not found at {master_path}")
        return

    # --- STEP 1: LOAD MASTER DATA & CALCULATE GLOBAL MEDIAN ---
    print(f"--- Step 1: Processing Master Dataset {master_path.name} ---")
    df_master = pd.read_csv(master_path)
    df_master['clean_ds_seq'] = df_master['sequence'].apply(clean_seq)
    
    # Identify indel columns (numeric headers)
    numeric_cols = [c for c in df_master.columns if str(c).lstrip('-').isdigit()]
    # Remove rows with no indel data
    df_master = df_master[(df_master[numeric_cols] != 0).any(axis=1)].copy()
    
    # Normalize indel counts to sum to 1.0 and calculate Experimental FS
    print("Normalizing ground truth indel distributions...")
    df_numeric = df_master[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_norm = df_numeric.div(df_numeric.sum(axis=1).replace(0, 1), axis=0)
    
    # FS = sum of all indels where size % 3 != 0
    fs_target_cols = [c for c in numeric_cols if int(c) % 3 != 0]
    df_master['experimental_fs'] = df_norm[fs_target_cols].sum(axis=1)
    
    # Calculate THE GLOBAL MEDIAN for the binary threshold
    global_median = df_master['experimental_fs'].median()
    print(f"Total Master Rows: {len(df_master)}")
    print(f"Global Median FS (Binary Threshold): {global_median:.6f}")

    # --- STEP 2: DISCOVER PREDICTOR FILES WITH SAME NAME ---
    print(f"\n--- Step 2: Searching for Predictors in subfolders (looking for '{dataset_filename}') ---")
    
    # merged_df starts with our ground truth data
    merged_df = df_master[['clean_ds_seq', 'experimental_fs']].copy()
    
    # Find all files named FORECasT_K562.csv in subfolders, excluding master path and results
    predictor_paths = []
    for f in root.rglob(dataset_filename):
        # Exclude the master file path itself and anything in tests/results
        if f.resolve() != master_path.resolve() and "tests" not in f.parts and "results" not in f.parts:
            predictor_paths.append(f)
            print(f"  Found predictor file: {f}")

    if not predictor_paths:
        print("Warning: No predictor files found with matching name in subfolders.")

    # Load and merge each predictor file
    for p_path in predictor_paths:
        try:
            p_df = pd.read_csv(p_path)
            if 'sequence' not in p_df.columns: continue
            
            # Identify columns to keep (must contain "FS")
            fs_cols = [c for c in p_df.columns if "FS" in str(c)]
            if not fs_cols: continue
            
            print(f"    - Merging {len(fs_cols)} predictors from {p_path.parent.name}...")
            p_df['clean_p_seq'] = p_df['sequence'].apply(clean_seq)
            
            # Keep only sequence and FS predictors
            p_subset = p_df[['clean_p_seq'] + fs_cols].copy()
            
            # Merge on sequence. Using 'left' to keep master rows.
            # If multiple models have same FS column names, we prefix with folder name
            prefix = p_path.parent.name
            p_subset.rename(columns={c: f"{prefix}_{c}" for c in fs_cols}, inplace=True)
            
            merged_df = pd.merge(
                merged_df, 
                p_subset.rename(columns={'clean_p_seq': 'clean_ds_seq'}), 
                on='clean_ds_seq', 
                how='left'
            )
        except Exception as e:
            print(f"    - Error loading predictor file {p_path}: {e}")

    # --- STEP 3: PROCESS TEST SPLITS ---
    print(f"\n--- Step 3: Evaluating AUC on Test Splits in {test_folder.name}/ ---")
    for test_file in test_folder.glob("*.csv"):
        print(f"\n>> Split: {test_file.name}")
        try:
            test_df = pd.read_csv(test_file)
            if 'sequence' not in test_df.columns: continue
            
            # Clean split sequences
            test_seqs = [clean_seq(s) for s in test_df['sequence'].dropna().unique()]
            
            # Suffix Matching: Filter merged_df for sequences that end with any test_seq
            # This aligns ground truth + predictors to the specific split
            def is_in_split(ds_seq):
                return any(ds_seq.endswith(ts) for ts in test_seqs)

            split_df = merged_df[merged_df['clean_ds_seq'].apply(is_in_split)].copy()

            if split_df.empty:
                print("   No sequence matches found.")
                continue

            # Binary target: 1 if experimental_fs > global_median
            y_true = (split_df['experimental_fs'] > global_median).astype(int)
            
            if len(y_true.unique()) < 2:
                print(f"   Skip: Split lacks binary diversity (Classes: {y_true.unique()})")
                continue

            # Identify all predictor columns (excluding the ground truth target)
            predictor_cols = [c for c in split_df.columns if "FS" in str(c) and c != 'experimental_fs']
            split_results = []

            for pred in predictor_cols:
                mask = split_df[pred].notna()
                if mask.any():
                    try:
                        # Calculate AUROC
                        score = roc_auc_score(y_true[mask], split_df.loc[mask, pred])
                        print(f"     [AUC] {pred}: {score:.4f}")
                        split_results.append({'Predictor': pred, 'AUC': score})
                    except Exception as e:
                        print(f"     [ERR] {pred}: {e}")

            # Save split results to the results folder
            if split_results:
                out_df = pd.DataFrame(split_results).set_index('Predictor')
                out_path = results_folder / f"auc_only_{test_file.name}"
                out_df.to_csv(out_path)
                print(f"   Saved results to {out_path}")

        except Exception as e:
            print(f"   Error processing split {test_file.name}: {e}")

analyze_k562_auc_final()

--- Step 1: Processing Master Dataset FORECasT_K562.csv ---
Normalizing ground truth indel distributions...
Total Master Rows: 35129
Global Median FS (Binary Threshold): 0.784042

--- Step 2: Searching for Predictors in subfolders (looking for 'FORECasT_K562.csv') ---
  Found predictor file: FORECasT_preds\FORECasT_K562.csv
  Found predictor file: inDelphi_preds\FORECasT_K562.csv
    - Merging 1 predictors from FORECasT_preds...
    - Merging 5 predictors from inDelphi_preds...

--- Step 3: Evaluating AUC on Test Splits in tests/ ---

>> Split: test_CROTON_APINDEL.csv
     [AUC] FORECasT_preds_FORECasT_FS_ratio: 0.8757
     [AUC] inDelphi_preds_inDelphi_FS_mESC: 0.8650
     [AUC] inDelphi_preds_inDelphi_FS_U2OS: 0.8610
     [AUC] inDelphi_preds_inDelphi_FS_HEK293: 0.8777
     [AUC] inDelphi_preds_inDelphi_FS_HCT116: 0.8769
     [AUC] inDelphi_preds_inDelphi_FS_K562: 0.8764
   Saved results to results\auc_only_test_CROTON_APINDEL.csv

>> Split: test_ex1_seed1.csv
     [AUC] FORECasT_pre